# Week 20, Day 5: The collections module

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w020d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from collections import Counter

votes = ["Ada", "Ben", "Ada", "Cy", "Ada", "Ben"]
c = Counter(votes)
print(c)
print(c["Ada"], c["Zoe"])
print(c.most_common(2))

In [ ]:
from collections import Counter

letters = Counter("mississippi")
print(letters)
print(letters.most_common(1), letters.total())

words = Counter()
for line in ["the cat sat", "the cat ran", "a dog sat"]:
    words.update(line.split())       # add each word's count
print(words.most_common(3))
words["the"] += 10
print(words["the"])

In [ ]:
from collections import Counter

stock = Counter(tea=5, jam=2, bread=4)
sold = Counter(tea=3, bread=4)
print(stock - sold)            # subtraction drops anything that reaches 0 or below
print(stock + sold)
print(Counter("listen") == Counter("silent"))    # same letters: an anagram
print(sorted(Counter("banana").elements()))

In [ ]:
from collections import defaultdict

words = ["apple", "avocado", "banana", "blueberry", "cherry", "apricot"]

by_letter = defaultdict(list)          # a missing key starts as []
for w in words:
    by_letter[w[0]].append(w)
print(dict(by_letter))

lengths = defaultdict(int)             # a missing key starts as 0
for w in words:
    lengths[len(w)] += 1
print(dict(lengths))

towns = defaultdict(set)               # a missing key starts as set()
for name, town in [("Ada", "Leeds"), ("Ben", "Bath"), ("Ada", "York"), ("Ada", "Leeds")]:
    towns[name].add(town)
print({k: sorted(v) for k, v in towns.items()})

> **Common mistake.** Just **reading** a missing key from a `defaultdict` creates it: `by_letter["z"]` adds an empty list under `"z"`. To check whether a key exists without creating it, use `"z" in by_letter`.

In [ ]:
from collections import deque

queue = deque(["Ada", "Ben"])
queue.append("Cy")               # join at the back
queue.appendleft("VIP")          # jump in at the front
print(queue)
print(queue.popleft(), queue.popleft())   # served from the front
print(queue, len(queue))

In [ ]:
from collections import deque

recent = deque(maxlen=3)
for reading in [21, 22, 25, 24, 30, 29]:
    recent.append(reading)
    print(list(recent), "average", round(sum(recent) / len(recent), 1))

In [ ]:
from collections import deque

players = deque(["Ada", "Ben", "Cy"])
for turn in range(4):
    print("turn", turn + 1, "goes to", players[0])
    players.rotate(-1)

In [ ]:
from collections import namedtuple

Member = namedtuple("Member", ["name", "age", "town"])

m = Member("Ada", 31, "Leeds")
print(m)
print(m.name, m.town, m[1])
name, age, town = m                  # still unpacks like a tuple
print(m._replace(town="York"))       # tuples can't change, so this makes a new one
print(m._asdict())

people = [Member("Ben", 25, "Bath"), m, Member("Cy", 40, "York")]
print([p.name for p in sorted(people, key=lambda p: p.age)])

In [ ]:
from collections import OrderedDict

cache = OrderedDict()
LIMIT = 3

def remember(key, value):
    cache[key] = value
    cache.move_to_end(key)             # most recently used goes to the end
    if len(cache) > LIMIT:
        cache.popitem(last=False)      # forget the least recently used

for k in ["a", "b", "c"]:
    remember(k, k.upper())
cache.move_to_end("a")                 # "a" was just used again
remember("d", "D")                     # full: "b" is forgotten
print(list(cache))

> **Key idea.** `Counter` for counting, `defaultdict` for grouping and accumulating, `deque` for queues and "the last n", `namedtuple` for readable fixed records, and `OrderedDict` when the order of use matters. All come from `from collections import ...`.

### Exercise: Word counts

Write `top_words(text, k)` returning the k most common words (lower case, split on spaces, ignoring the punctuation `.,!?` at the ends of words) as `(word, count)` pairs, using `Counter`. Ties keep the order of first appearance, which `most_common` does for you.

In [ ]:
from collections import Counter

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    t = \"The cat sat. The cat ran! A dog sat, the end.\"\n    assert top_words(t, 2) == [(\"the\", 3), (\"cat\", 2)]\n    assert top_words(t, 4) == [(\"the\", 3), (\"cat\", 2), (\"sat\", 2), (\"ran\", 1)]\n    assert top_words(\"\", 3) == []")

### Exercise: Stock and orders

Write `can_fulfil(stock, order)` that returns `True` if every item in `order` (a list of item names, with repeats) is available in `stock` (a dictionary of counts), and `left_after(stock, order)` returning a plain dictionary of what's left (items that reach 0 are dropped). Use `Counter` arithmetic, not loops.

In [ ]:
from collections import Counter

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    stock = {\"tea\": 3, \"jam\": 1, \"bread\": 2}\n    assert can_fulfil(stock, [\"tea\", \"tea\", \"jam\"]) is True\n    assert can_fulfil(stock, [\"jam\", \"jam\"]) is False and can_fulfil(stock, [\"milk\"]) is False\n    assert left_after(stock, [\"tea\", \"jam\"]) == {\"tea\": 2, \"bread\": 2}\n    assert \"for \" not in source")

### Exercise: Group by

Write `group_by(items, key)` returning a plain dictionary from each `key(item)` to the list of items with that key, in their original order, using `defaultdict(list)`. Then use it to set `by_length` (the words in `WORDS` grouped by length) and `by_first` (grouped by first letter, lower case).

In [ ]:
from collections import defaultdict

WORDS = ["Kiwi", "fig", "date", "Apple", "lime", "pear", "avocado"]

by_length = ___
by_first = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert by_length == {4: [\"Kiwi\", \"date\", \"lime\", \"pear\"], 3: [\"fig\"], 5: [\"Apple\"], 7: [\"avocado\"]}\n    assert by_first == {\"k\": [\"Kiwi\"], \"f\": [\"fig\"], \"d\": [\"date\"], \"a\": [\"Apple\", \"avocado\"], \"l\": [\"lime\"], \"p\": [\"pear\"]}\n    assert group_by([], len) == {} and type(group_by([1], str)) is dict")

### Exercise: A printer queue

Write `run_queue(jobs)` where each job is `(name, pages)`, and the printer does at most 5 pages of a job per turn: if a job has more left, the rest goes to the **back** of the queue. Return the list of names in the order their jobs **finish**. Use a `deque`.

In [ ]:
from collections import deque

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert run_queue([(\"A\", 3), (\"B\", 12), (\"C\", 5), (\"D\", 6)]) == [\"A\", \"C\", \"D\", \"B\"]\n    assert run_queue([]) == [] and run_queue([(\"X\", 20)]) == [\"X\"]")

### Exercise: A moving average

Write `moving_averages(readings, n)` returning, for each reading, the average of the last `n` readings up to and including it (fewer at the start), rounded to 2 decimal places. Use `deque(maxlen=n)` and keep a running total, so each step is O(1).

In [ ]:
from collections import deque

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert moving_averages([21, 22, 25, 24, 30, 29], 3) == [21.0, 21.5, 22.67, 23.67, 26.33, 27.67]\n    assert moving_averages([], 3) == [] and moving_averages([5, 7], 1) == [5.0, 7.0]\n    import random\n    data = [random.Random(1).randint(0, 100) for _ in range(100_000)]\n    assert len(moving_averages(data, 50)) == 100_000")

### Bug hunt: Calling the factory

This crashes with `TypeError: first argument must be callable or None`. Fix it so it prints `{'a': ['ant', 'ape'], 'b': ['bee']}`.

In [ ]:
from collections import defaultdict

groups = defaultdict(list())
for w in ["ant", "bee", "ape"]:
    groups[w[0]].append(w)
print(dict(groups))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"{'a': ['ant', 'ape'], 'b': ['bee']}\")")

### Bug hunt: Tuples don't change

This crashes with `AttributeError: can't set attribute`, because namedtuples can't be changed. Fix it, without changing the first three lines, so it prints `Point(x=3, y=10)`.

In [ ]:
from collections import namedtuple
Point = namedtuple("Point", ["x", "y"])
p = Point(3, 4)
p.y = 10
print(p)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"Point(x=3, y=10)\")")

### Puzzle: Count, then rank

Arrange the lines so the program prints the two most common colours. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them.

In [ ]:
a = ___   # Counter("aab")["c"]
b = ___   # list(deque([1, 2, 3], maxlen=2))
c = ___   # Counter(a=3, b=1) - Counter(a=1, b=2), as a plain dict
d = ___   # namedtuple("P", ["x", "y"])(1, 2)[1]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == 0 and b == [2, 3] and c == {\"a\": 2} and d == 2")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: shortest route through a maze

In Week 16 you found **a** path through a maze with backtracking, but not the shortest. Breadth-first search (BFS) finds the shortest: explore squares in order of distance from the start, using a `deque` as the queue of squares still to visit. Write `shortest(maze)` returning the number of steps on the shortest path from `"S"` to `"E"` (up, down, left, right, never through `"#"`), or `-1` if there is none, and `path(maze)` returning the list of squares `(row, col)` on one shortest path, start to end (empty if none).

Keep a dictionary `came_from` from each square reached to the square you reached it from; following it backwards from `"E"` gives the path.

In [ ]:
from collections import deque

In [ ]:
# Check: run the exercise cell above first, then this one.
check("M = [\"S.#.....\",\n     \".##.###.\",\n     \"....#...\",\n     \"###.#.#.\",\n     \"....#.#E\"]\ndef test_shortest():\n    \"shortest\"\n    assert shortest(M) == 15\n    assert shortest([\"S#E\"]) == -1 and shortest([\"SE\"]) == 1\ndef test_path():\n    \"path\"\n    p = path(M)\n    assert p[0] == (0, 0) and p[-1] == (4, 7) and len(p) == 16\n    for (r1, c1), (r2, c2) in zip(p, p[1:]):\n        assert abs(r1 - r2) + abs(c1 - c2) == 1 and M[r2][c2] != \"#\"\n    assert path([\"S#E\"]) == []")

## Warm-up

One line does what took a loop and an `if` in Week 10. And asking for a name that was never counted gives 0, not a `KeyError`.

## Counter

A `Counter` is a dictionary from things to how many times they occurred. Make one from any iterable (a list, a string, a generator), or add counts as you go:

Counters can be added, subtracted and compared:

- `most_common(k)` gives the k most common `(item, count)` pairs; with no argument, all of them, most common first.
- `total()` adds up all the counts.
- `elements()` repeats each item as many times as its count.
- Comparing two Counters for equality is the quickest anagram test there is.

## defaultdict

A `defaultdict` is a dictionary that creates a missing key **automatically**, with a starting value from a function you give it. It's built for grouping and accumulating:

The argument is a **function that makes the starting value**: `list`, `int`, `set`, or your own (`defaultdict(lambda: "unknown")`). Pass the function itself, without brackets.

Compare the same grouping without it:

```
groups = {}
for w in words:
    if w[0] not in groups:
        groups[w[0]] = []
    groups[w[0]].append(w)
```

(`groups.setdefault(w[0], []).append(w)` does the same in one line on an ordinary dictionary, without the import.)

## deque

A `deque` (pronounced "deck", short for "double-ended queue") is a list-like container that's fast at **both ends**. Adding or removing at the front of a list is O(n), because everything else shifts along (Week 17); on a deque it's O(1).

Use a deque for any **queue**: jobs waiting to be done, people waiting in line, squares to visit in a search (you'll use this for breadth-first search in Phase 3).

**`maxlen`** makes a deque that keeps only the most recent items: when it's full, adding at one end drops an item off the other. Perfect for "the last n":

`rotate(k)` moves items round in a circle, k places to the right (or left, if negative), which is handy for taking turns:

Deques support indexing (`players[0]`), but indexing the middle of a long deque is O(n); if you need fast access by position, use a list.

## namedtuple

A tuple like `("Ada", 31, "Leeds")` works, but `person[2]` doesn't say what it means. A **namedtuple** gives the positions names, while still being a tuple:

`namedtuple("Member", ["name", "age", "town"])` makes a new **type**, conventionally given a capitalised name. Records made from it are immutable, compare and sort like tuples, and take no more memory than plain tuples. When you need records that can change, or that have their own behaviour, you'll write classes (Phase 3), and Python's `dataclasses` make that just as quick.

## OrderedDict

Ordinary dictionaries keep keys in the order they were added, so you rarely need `OrderedDict` for that any more. What it adds is **reordering**: `move_to_end(key)` moves a key to the end (or the start, with `last=False`), and `popitem(last=False)` removes the **oldest**. Together they make a **least recently used (LRU) cache**: a memo with limited room, which forgets whatever was used longest ago:

This is exactly what `functools.lru_cache(maxsize=...)` from Week 16 does for you.

## Practice

## Quiz

## Daily challenge

## Before you go

Go back to two programs from Weeks 10 and 11 that counted or grouped things with plain dictionaries, and rewrite them with `Counter` or `defaultdict`. Count how many lines you saved. Tomorrow is the Forge: a small program built from your own modules and this week's standard library.